# USD Rates RV v2 — Rolling PCA fly screener

Showcases the v2 RV toolkit additions on ~1y EOD SOFR data:
- **Rolling PCA residual** with eigenvector continuity (`rolling_residual`, `align_eigenvectors`)
- **Optimal OU bands** — Zeng-Lee 2014 (`optimal_ou_thresholds`)
- **OU S-score** — Avellaneda-Lee (`ou_sscore`)
- **ADF gate** (`adf_gate`)
- **Eigenportfolio returns** (`eigenportfolio_returns`)
- **Lead-lag** — Lévy area + cross-correlation (`levy_area`, `xcorr_lead_lag`)
- **Cost-aware screener** (`make_pca_fly_rv_screener`)

Mirrors `rv_pca_curve_fly.ipynb` plumbing.

In [ ]:
%matplotlib inline
import sys; sys.path.append("../../")
import datetime, pytz, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (15, 6); plt.rcParams["axes.grid"] = True
from RVUtils.plt_timeseries import make_secondary_axis_plot

NYC = pytz.timezone("America/New_York")
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")
ts = TimeseriesBuilder()
start = NYC.localize(datetime.datetime(2025, 6, 2, 17, 0))
end = NYC.localize(datetime.datetime(2026, 6, 24, 17, 0))
ROUTERS = {"IRS": IRSwapsTB(curve_mdp, show_tqdm=False), "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=False)}

def load(queries):
    return ts.get_timeseries(start=start, end=end, queries=queries, n_jobs=8, routers=ROUTERS)

def sofr(tenors):
    df = load([UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_RATE) for t in tenors])
    return df.rename(columns={f"USD-SOFR-1D {t} OUTRIGHT RATE": t for t in tenors})[list(tenors)]

print("Setup OK. Window:", start.date(), "->", end.date())

In [ ]:
TENORS = ["2y", "3y", "5y", "7y", "10y", "20y", "30y"]
df_s = sofr(TENORS)
print(f"SOFR curve: {df_s.shape[0]} rows x {df_s.shape[1]} tenors")
df_s.tail(3)

## 1. Rolling PCA residual with eigenvector continuity (spec A)

Re-fits PCA on a trailing 261-day window at each step. `align_eigenvectors` flips sign of eigenvectors
across windows to prevent spurious jumps in the residual.

In [ ]:
from RVUtils.pca_rv import rolling_residual, make_pca_rv_builder, eigenportfolio_returns

# Rolling PCA residual for outright 10y
resid_10y = rolling_residual(df_s, "10y", window=200, k=3, sign_align=True)
print(f"Rolling residual: {resid_10y.notna().sum()} observations")

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="matplotlib", title="10y SOFR: rolling PCA residual (200d window, k=3)"
)
plot(resid_10y.rename("10y rolling PCA residual") * 100, which="left")
z = (resid_10y - resid_10y.rolling(65).mean()) / resid_10y.rolling(65).std()
plot(z.rename("z-score (65d)"), which="right",
     indicators=[{"kind": "zbands", "entry": 2, "stop": 3}])
legend(show_date=True, loc="upper left")
plt.show()

In [ ]:
# Rolling PCA residual for 2s5s10s fly
fit_fn, fv_fn, res_fn, fly_w_fn, *_ = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)
fit_fn()
w_2s5s10s = fly_w_fn("2y", "5y", "10y")
print("PCA-neutral 2s5s10s weights:", {k: round(v, 3) for k, v in w_2s5s10s.items()})

resid_fly = rolling_residual(df_s, ["2y", "5y", "10y"], weights=w_2s5s10s, window=200, k=3)

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="matplotlib", title="2s5s10s SOFR fly: rolling PCA residual"
)
plot(resid_fly.rename("2s5s10s rolling residual") * 100, which="left")
z_fly = (resid_fly - resid_fly.rolling(65).mean()) / resid_fly.rolling(65).std()
plot(z_fly.rename("z-score (65d)"), which="right",
     indicators=[{"kind": "zbands", "entry": 2, "stop": 3}])
legend(show_date=True, loc="upper left")
plt.show()

## 2. OU calibration + optimal entry/exit bands (spec B)

Calibrate OU on the rolling residual, then compute Zeng-Lee optimal thresholds.

In [ ]:
from RVUtils.mean_reversion import calibrate_ou, optimal_ou_thresholds, ou_sscore, adf_gate

# OU calibration on 2s5s10s rolling residual
ou = calibrate_ou(resid_fly)
print("OU calibration (2s5s10s rolling residual):")
for k in ["mu", "kappa", "sigma", "half_life"]:
    print(f"  {k}: {ou[k]:.4f}")

# Optimal thresholds (in sigma_eq units)
from RVUtils.cost_model import structure_cost_bps
cost_bp = structure_cost_bps([2, 5, 10], [abs(w_2s5s10s["2y"]), abs(w_2s5s10s["5y"]), abs(w_2s5s10s["10y"])])
sigma_eq = ou["sigma"] / np.sqrt(2 * ou["kappa"]) if ou["kappa"] > 0 else np.nan
cost_sigma = (cost_bp / 10000) / sigma_eq if np.isfinite(sigma_eq) and sigma_eq > 0 else 0.0

a_sym, b_sym = optimal_ou_thresholds(ou["kappa"], ou["sigma"], cost=cost_sigma, case="symmetric")
a_lo, b_lo = optimal_ou_thresholds(ou["kappa"], ou["sigma"], cost=cost_sigma, case="long_only")
print(f"\nStructure cost: {cost_bp:.2f} bp -> {cost_sigma:.4f} sigma_eq units")
print(f"Optimal symmetric bands: entry ±{a_sym:.3f}σ, exit ±{-b_sym:.3f}σ")
print(f"Optimal long-only bands: entry {a_lo:.3f}σ, exit {b_lo:.3f}σ")

## 3. OU S-score + ADF gate (spec D, E)

S-score standardizes the residual by the OU equilibrium volatility. ADF gate filters non-stationary residuals.

In [ ]:
# S-score on multiple fly residuals
flies = {
    "2s5s10s": (["2y", "5y", "10y"], w_2s5s10s),
    "5s10s30s": (["5y", "10y", "30y"], fly_w_fn("5y", "10y", "30y")),
    "2s10s30s": (["2y", "10y", "30y"], fly_w_fn("2y", "10y", "30y")),
    "3s7s20s": (["3y", "7y", "20y"], fly_w_fn("3y", "7y", "20y")),
}

rows = []
for name, (legs, wts) in flies.items():
    resid = rolling_residual(df_s, legs, weights=wts, window=200, k=3)
    if resid.notna().sum() < 30:
        continue
    sscore = ou_sscore(resid)
    adf_ok = adf_gate(resid, pval=0.10)
    ou_p = calibrate_ou(resid)
    rows.append({
        "fly": name,
        "s_score": round(float(sscore.iloc[-1]), 3) if sscore.notna().sum() > 0 else np.nan,
        "adf_pass": adf_ok,
        "half_life": round(ou_p["half_life"], 1),
        "kappa": round(ou_p["kappa"], 4),
        "weights": {k: round(v, 3) for k, v in wts.items()},
    })

df_sscore = pd.DataFrame(rows).set_index("fly")
print("S-score + ADF gate for SOFR flies (rolling PCA residual):")
df_sscore

## 4. Eigenportfolio returns (spec F)

Factor-mimicking portfolio returns: `F_j(t) = Σ_i (v_ji / σ_i) R_i(t)`.

In [ ]:
_, get_model_fn = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)[-2:]
# re-fit to get the model
fit2, *_, get_model2 = make_pca_rv_builder(df_s, on="levels", n_factors=3, sort_by_tenor=False)
fit2()
model, _ = get_model2()

asset_returns = df_s.diff().dropna()
asset_vols = asset_returns.std()
F = eigenportfolio_returns(model.loadings[["PC1", "PC2", "PC3"]], asset_returns, asset_vols)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for i, pc in enumerate(["PC1", "PC2", "PC3"]):
    axes[i].plot(F[pc].cumsum(), label=f"{pc} eigenportfolio (cumulative)")
    axes[i].legend(loc="upper left")
    axes[i].grid(True)
axes[0].set_title("Eigenportfolio cumulative returns — SOFR curve")
plt.tight_layout()
plt.show()

print("Eigenportfolio return correlation matrix:")
F.corr().round(3)

## 5. Lead-lag — Lévy area + cross-correlation (spec C)

Does the 5y or 10y leg lead the other? Useful for execution timing.

In [ ]:
from RVUtils.lead_lag import levy_area, xcorr_lead_lag

# Lead-lag between 5y and 10y
ll = xcorr_lead_lag(df_s["5y"], df_s["10y"], max_lag=5)
print(f"5y vs 10y cross-corr lead-lag: lag={ll['lag']} (positive => 5y leads), corr={ll['corr']:.4f}")

ll2 = xcorr_lead_lag(df_s["2y"], df_s["30y"], max_lag=5)
print(f"2y vs 30y cross-corr lead-lag: lag={ll2['lag']}, corr={ll2['corr']:.4f}")

la = levy_area(df_s["5y"], df_s["10y"], window=20)
plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    engine="matplotlib", title="Lévy area: 5y vs 10y SOFR (>0 => 5y leads)"
)
plot(la.rename("Lévy area (20d)"), which="left")
legend(show_date=True)
plt.show()

## 6. Cost-aware fly screener — `make_pca_fly_rv_screener` (spec J)

End-to-end *Catching the Butterfly* pipeline: rolling PCA residual → ADF gate → OU half-life → cost-aware composite ranking.

In [ ]:
from RVUtils.screener_rv import make_pca_fly_rv_screener

structures = {
    "2s5s10s": ("2y", "5y", "10y"),
    "2s5s30s": ("2y", "5y", "30y"),
    "2s10s30s": ("2y", "10y", "30y"),
    "3s5s10s": ("3y", "5y", "10y"),
    "3s7s10s": ("3y", "7y", "10y"),
    "3s7s20s": ("3y", "7y", "20y"),
    "5s7s10s": ("5y", "7y", "10y"),
    "5s10s30s": ("5y", "10y", "30y"),
    "7s10s30s": ("7y", "10y", "30y"),
    "7s20s30s": ("7y", "20y", "30y"),
}

# Get PCA-neutral weights for each fly
weights_map = {}
for name, (s, b, l) in structures.items():
    try:
        w = fly_w_fn(s, b, l)
        weights_map[name] = w
    except Exception as e:
        print(f"Skipping {name}: {e}")

result = make_pca_fly_rv_screener(
    df_s,
    structures,
    weights_map,
    window=200,
    k=3,
    cost_z=0.0,
    lambda_carry=0.0,
    adf_pval=0.10,
)

print(f"Screener: {len(result)} structures ranked by |composite|")
result.style.format({
    "level": "{:.4f}",
    "zscore": "{:.2f}",
    "percentile": "{:.2f}",
    "vol": "{:.4f}",
    "half_life": "{:.1f}",
    "composite": "{:.4f}",
    "carry": "{:.4f}",
}).background_gradient(subset=["composite"], cmap="RdYlGn_r")

## 7. Deep dive — top-ranked fly

Rolling residual, S-score, OU forecast, optimal bands for the top-ranked structure.

In [ ]:
from RVUtils.mean_reversion import ou_conditional

top_name = result.index[0]
top_legs = list(structures[top_name])
top_wts = weights_map[top_name]
print(f"Top fly: {top_name}  weights: {{{', '.join(f'{k}: {v:.3f}' for k, v in top_wts.items())}}}")
print(f"  direction: {result.loc[top_name, 'direction']}  z: {result.loc[top_name, 'zscore']:.2f}  hl: {result.loc[top_name, 'half_life']:.1f}d  adf: {result.loc[top_name, 'adf_pass']}")

top_resid = rolling_residual(df_s, top_legs, weights=top_wts, window=200, k=3)
top_ou = calibrate_ou(top_resid)
top_sscore = ou_sscore(top_resid)

# OU conditional forecast
x0 = float(top_resid.iloc[-1])
cond = ou_conditional(x0, top_ou, horizon=top_ou["half_life"])
print(f"  OU forecast ({top_ou['half_life']:.0f}d): current={x0*100:.2f}bp -> E[x]={cond['mean']*100:.2f}bp ± {cond['std']*100:.2f}bp")

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
axes[0].plot(top_resid.index, top_resid.values * 100, label=f"{top_name} rolling PCA residual (bp)")
axes[0].axhline(top_ou["mu"] * 100, color="red", ls="--", label=f"OU μ = {top_ou['mu']*100:.2f}bp")
axes[0].legend(loc="upper left")
axes[0].grid(True)
axes[0].set_title(f"{top_name}: rolling PCA residual + OU mean")

axes[1].plot(top_sscore.index, top_sscore.values, label="S-score", color="tab:orange")
axes[1].axhline(1.75, color="green", ls="--", alpha=0.7, label="entry ±1.75")
axes[1].axhline(-1.75, color="green", ls="--", alpha=0.7)
axes[1].axhline(0.75, color="gray", ls=":", alpha=0.7, label="exit ±0.75")
axes[1].axhline(-0.75, color="gray", ls=":", alpha=0.7)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].legend(loc="upper left")
axes[1].grid(True)
axes[1].set_title(f"{top_name}: OU S-score (Avellaneda-Lee)")

plt.tight_layout()
plt.show()

## 8. OLS sub-period beta stability (spec I)

Check whether the PCA fly's beta to level/slope has been stable across regimes.

In [ ]:
from RVUtils.regression import ols_segment

# Build a fly timeseries and regress on level (PC1 score) and slope (PC2 score)
_, scores = get_model2()
fly_ts = sum(top_wts[c] * df_s[c] for c in top_legs).dropna()
fly_ts.name = top_name

common = fly_ts.index.intersection(scores.index)
mid = common[len(common) // 2]
periods = [(common[0], mid), (mid, common[-1])]

seg = ols_segment(
    fly_ts.loc[common],
    scores[["PC1", "PC2"]].loc[common],
    periods,
)

print(f"Beta stability for {top_name} vs PC1/PC2:")
for s in seg:
    p0, p1 = pd.Timestamp(s["period"][0]), pd.Timestamp(s["period"][1])
    print(f"  {p0.date()} -> {p1.date()}: "
          f"β_PC1={s['betas']['PC1']:.4f}, β_PC2={s['betas']['PC2']:.4f}, "
          f"adj-R²={s['adj_r2']:.3f} (n={s['nobs']})")